# 1. Setup

### 1.1 Install deps & packages

In [1]:
%pip install numpy pandas matplotlib kagglehub
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import kagglehub
import shutil
import os


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/kaneviggers/Desktop/Transaction-fraud-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Download dataset

In [2]:
try:
    os.mkdir('original_dataset')
    kagglehub.dataset_download("zahranusratt/banking-fraud-detection-dataset", output_dir='original_dataset')
except FileExistsError:
    pass

# Remove uesless metadata
shutil.rmtree('original_dataset/.complete/', ignore_errors=True)

print('Dataset downloaded')

Dataset downloaded


# 2. Data pipeline

### 2.1 Load csv into dataframe

In [3]:
fraud_data = pd.read_csv('original_dataset/bank_fraud.csv')

print(f"Shape\n{fraud_data.shape}\n")
print(f"Head\n{fraud_data.head()}")
print("\nColumns")
for column_number, column_name in enumerate(fraud_data.columns, start=1):
    print(column_number, column_name)

Shape
(50000, 25)

Head
   Transaction_ID  Customer_ID  Transaction_Amount (in Million)  \
0        431438.0      24239.0                              6.0   
1        902451.0      77250.0                              9.0   
2        223410.0      34294.0                              3.0   
3        145626.0      92041.0                              1.0   
4        414637.0      71578.0                              1.0   

  Transaction_Time Transaction_Date Transaction_Type  Merchant_ID  \
0            10:54       2025-03-08              POS      97028.0   
1            19:23       2025-01-17              ATM      27515.0   
2            10:20       2025-04-30              POS      13810.0   
3            14:11       2025-02-21           Online      10501.0   
4            04:12       2025-04-11           Online      53569.0   

  Merchant_Category Transaction_Location Customer_Home_Location  ...  \
0               ATM            Singapore                 Lahore  ...   
1             

In [4]:
fraud_data_original = fraud_data.copy(deep=True)

print("Records:", fraud_data_original.shape[0])
print("Columns:", fraud_data_original.shape[1])

print("\nData-type counts:")
print(fraud_data_original.dtypes.value_counts())

Records: 50000
Columns: 25

Data-type counts:
float64    13
object     12
Name: count, dtype: int64


In [5]:
summary = pd.DataFrame({
    'dtype': fraud_data_original.dtypes,
    'sample_values': [fraud_data_original[col].dropna().unique()[:3] for col in fraud_data_original.columns]
})
print(summary.to_string())

                                         dtype                                  sample_values
Transaction_ID                         float64                 [431438.0, 902451.0, 223410.0]
Customer_ID                            float64                    [24239.0, 77250.0, 34294.0]
Transaction_Amount (in Million)        float64                                [6.0, 9.0, 3.0]
Transaction_Time                        object                          [10:54, 19:23, 10:20]
Transaction_Date                        object           [2025-03-08, 2025-01-17, 2025-04-30]
Transaction_Type                        object                             [POS, ATM, Online]
Merchant_ID                            float64                    [97028.0, 27515.0, 13810.0]
Merchant_Category                       object                    [ATM, Electronics, Grocery]
Transaction_Location                    object                [Singapore, Faisalabad, London]
Customer_Home_Location                  object              

2.  Numeric ranges, missing values, duplicates, and inconsistent entries

In [6]:


print("\n" + "="*60)
print("NUMERIC RANGES")
print("="*60)
print(fraud_data_original.describe())

print("\n" + "="*60)
print("CATEGORICAL SUMMARY")
print("="*60)
print(fraud_data_original.describe(include='object'))

print("\n" + "="*60)
print("UNIQUE VALUES (categorical columns)")
print("="*60)
for col in fraud_data_original.select_dtypes(include="object").columns:
    print(f"\n{col}: {fraud_data_original[col].nunique()} unique values")
    print(fraud_data_original[col].unique()[:15])

print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
missing = fraud_data_original.isna().sum()
missing_pct = (missing / len(fraud_data_original) * 100).round(2)
missing_summary = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(missing_summary[missing_summary["missing"] > 0].sort_values("missing", ascending=False))

print("\n" + "="*60)
print("DUPLICATE ROWS")
print("="*60)
print("Full duplicate rows:", fraud_data_original.duplicated().sum())
if "transaction_id" in fraud_data_original.columns:
    print("Duplicate transaction_id:", fraud_data_original["transaction_id"].duplicated().sum())


NUMERIC RANGES
       Transaction_ID   Customer_ID  Transaction_Amount (in Million)  \
count    49997.000000  49990.000000                     49991.000000   
mean    550400.968898  54869.720744                         4.999880   
std     259677.602349  26052.824933                         2.582025   
min     100043.000000  10005.000000                         1.000000   
25%     324445.000000  32259.250000                         3.000000   
50%     552115.000000  54720.500000                         5.000000   
75%     775942.000000  77542.000000                         7.000000   
max     999992.000000  99996.000000                         9.000000   

        Merchant_ID  Distance_From_Home      Device_ID  \
count  49993.000000        49998.000000   49991.000000   
mean   54951.375913          300.098564  552563.600088   
std    25983.342481          172.848263  260186.451027   
min    10001.000000            1.000000  100053.000000   
25%    32545.000000          150.000000  3276

## 3.1 Dropping columns

Looking at the dataset, we can see the transaction ID is unique to every row, therefore it won't be useful in deriving any value. We can use the customer contry to identify if the transaction took place in a high risk country, for this reason we can drop the city as it's too spesific.

In [7]:
fraud_data = fraud_data.drop(columns=['Transaction_ID'])

fraud_data.head()

,Customer_ID,Transaction_Amount (in Million),Transaction_Time,Transaction_Date,Transaction_Type,Merchant_ID,Merchant_Category,Transaction_Location,Customer_Home_Location,Distance_From_Home,...,Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,Unusual_Time_Transaction,Previous_Fraud_Count,Fraud_Label
0,24239.0,6.0,10:54,2025-03-08,POS,97028.0,ATM,Singapore,Lahore,466.0,...,4.0,17.0,2.0,4.0,Yes,Yes,0.0,No,1.0,Normal
1,77250.0,9.0,19:23,2025-01-17,ATM,27515.0,ATM,Singapore,Lahore,215.0,...,4.0,9.0,5.0,8.0,Yes,Yes,1.0,No,1.0,Normal
2,34294.0,3.0,10:20,2025-04-30,POS,13810.0,Electronics,Faisalabad,Faisalabad,216.0,...,5.0,18.0,5.0,8.0,Yes,No,0.0,Yes,1.0,Normal
3,92041.0,1.0,14:11,2025-02-21,Online,10501.0,Grocery,London,Karachi,408.0,...,6.0,18.0,5.0,1.0,No,Yes,2.0,Yes,1.0,Normal
4,71578.0,1.0,04:12,2025-04-11,Online,53569.0,Electronics,Singapore,Islamabad,209.0,...,3.0,18.0,4.0,3.0,No,Yes,1.0,No,1.0,Normal


Fraud_label is uencoded as 0 for normal and 1 for fraud, the mean gives the proportion of fraudulent transactions within each category.

In [8]:
fraud_data['Fraud_Label'] = fraud_data['Fraud_Label'].map({'Normal': 0, 'Fraud': 1})

In [9]:
for col in ['Merchant_Category', 'Transaction_Type', 'Card_Type']:
    rates = fraud_data.groupby(col)['Fraud_Label'].mean().sort_values(ascending=False)
    rates.name = 'Fraud_Rate'
    print(rates)
    print()

Merchant_Category
Restaurant     0.050342
ATM            0.050119
Fuel           0.048821
Grocery        0.048110
Electronics    0.047352
Clothing       0.045888
Name: Fraud_Rate, dtype: float64

Transaction_Type
Online    0.050383
ATM       0.047782
POS       0.047169
Name: Fraud_Rate, dtype: float64

Card_Type
Credit    0.048857
Debit     0.048080
Name: Fraud_Rate, dtype: float64



### 3.2 Normalising data

Since a lot of these columns are categories we can convert them to a number using a dict

In [10]:
categorical_columns = ['Merchant_Category', 'Transaction_Type', 'Card_Type']

category_values = {}

for col in categorical_columns:
    categories = fraud_data[col].astype('category').cat.categories.tolist()
    category_values[col] = categories
    fraud_data[col] = fraud_data[col].astype('category').cat.codes

# Show the mapping
for col, cats in category_values.items():
    print(col, dict(enumerate(cats)))

fraud_data.head()

Merchant_Category {0: 'ATM', 1: 'Clothing', 2: 'Electronics', 3: 'Fuel', 4: 'Grocery', 5: 'Restaurant'}
Transaction_Type {0: 'ATM', 1: 'Online', 2: 'POS'}
Card_Type {0: 'Credit', 1: 'Debit'}


,Customer_ID,Transaction_Amount (in Million),Transaction_Time,Transaction_Date,Transaction_Type,Merchant_ID,Merchant_Category,Transaction_Location,Customer_Home_Location,Distance_From_Home,...,Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,Unusual_Time_Transaction,Previous_Fraud_Count,Fraud_Label
0,24239.0,6.0,10:54,2025-03-08,2,97028.0,0,Singapore,Lahore,466.0,...,4.0,17.0,2.0,4.0,Yes,Yes,0.0,No,1.0,0.0
1,77250.0,9.0,19:23,2025-01-17,0,27515.0,0,Singapore,Lahore,215.0,...,4.0,9.0,5.0,8.0,Yes,Yes,1.0,No,1.0,0.0
2,34294.0,3.0,10:20,2025-04-30,2,13810.0,2,Faisalabad,Faisalabad,216.0,...,5.0,18.0,5.0,8.0,Yes,No,0.0,Yes,1.0,0.0
3,92041.0,1.0,14:11,2025-02-21,1,10501.0,4,London,Karachi,408.0,...,6.0,18.0,5.0,1.0,No,Yes,2.0,Yes,1.0,0.0
4,71578.0,1.0,04:12,2025-04-11,1,53569.0,2,Singapore,Islamabad,209.0,...,3.0,18.0,4.0,3.0,No,Yes,1.0,No,1.0,0.0


In [11]:
import re

def is_valid_ipv4(ip):
    if not isinstance(ip, str):
        return False
    match = re.match(r'^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})$', ip)
    if not match:
        return False
    return all(0 <= int(octet) <= 255 for octet in match.groups())

before = len(fraud_data)
fraud_data = fraud_data[fraud_data['IP_Address'].apply(is_valid_ipv4)]
dropped = before - len(fraud_data)
print(f"Dropped {dropped} rows with malformed IP addresses ({len(fraud_data)} remaining)")

Dropped 6 rows with malformed IP addresses (49994 remaining)


### 3.3 Flagging potentially risky transactions

A transaction is flagged as `potentially_risky` when 2 or more of the following signals are present:
- Is an international transaction
- Is with a new merchant
- Occurs at an unusual time
- Has a failed transaction count > 0
- Has a previous fraud history
- Distance from home > 400
- Transaction amount ≥ 7 million

In [19]:
risk_conditions = (
    (fraud_data['Is_International_Transaction'] == 'Yes').astype(int) +
    (fraud_data['Is_New_Merchant'] == 'Yes').astype(int) +
    (fraud_data['Unusual_Time_Transaction'] == 'Yes').astype(int) +
    (fraud_data['Failed_Transaction_Count'] > 0).astype(int) +
    (fraud_data['Previous_Fraud_Count'] > 0).astype(int) +
    (fraud_data['Distance_From_Home'] > 1000).astype(int) +
    (fraud_data['Transaction_Amount (in Million)'] >= 7).astype(int)
)

fraud_data['potentially_risky'] = (risk_conditions >= 2).astype(int)

risky_count = fraud_data['potentially_risky'].sum()
print(f"Flagged {risky_count} potentially risky transactions ({risky_count / len(fraud_data) * 100:.1f}%)")
print(f"\nFraud rate in risky transactions:     {fraud_data[fraud_data['potentially_risky'] == 1]['Fraud_Label'].mean():.3f}")
print(f"Fraud rate in non-risky transactions: {fraud_data[fraud_data['potentially_risky'] == 0]['Fraud_Label'].mean():.3f}")
fraud_data['potentially_risky'].value_counts()

Flagged 24950 potentially risky transactions (49.9%)

Fraud rate in risky transactions:     0.050
Fraud rate in non-risky transactions: 0.047


potentially_risky
0    25044
1    24950
Name: count, dtype: int64

### 3.4 Vectorisation of remaining features

Here we convert text based columns into numrical values.

In [ ]:
binary_cols = ['Is_International_Transaction', 'Is_New_Merchant', 'Unusual_Time_Transaction']
for col in binary_cols:
    fraud_data[col] = fraud_data[col].map({'Yes': 1, 'No': 0})

location_cols = ['Transaction_Location', 'Customer_Home_Location']
location_values = {}

for col in location_cols:
    categories = fraud_data[col].astype('category').cat.categories.tolist()
    location_values[col] = categories
    fraud_data[col] = fraud_data[col].astype('category').cat.codes

for col, cats in location_values.items():
    print(col, dict(enumerate(cats)))

fraud_data.head()

Transaction_Location {0: 'Bangkok', 1: 'Dubai', 2: 'Faisalabad', 3: 'Islamabad', 4: 'Karachi', 5: 'Kuala Lumpur', 6: 'Lahore', 7: 'London', 8: 'Multan', 9: 'Singapore'}
Customer_Home_Location {0: 'Faisalabad', 1: 'Islamabad', 2: 'Karachi', 3: 'Lahore', 4: 'Multan'}


,Customer_ID,Transaction_Amount (in Million),Transaction_Time,Transaction_Date,Transaction_Type,Merchant_ID,Merchant_Category,Transaction_Location,Customer_Home_Location,Distance_From_Home,...,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,Unusual_Time_Transaction,Previous_Fraud_Count,Fraud_Label,potentially_risky
0,24239.0,6.0,10:54,2025-03-08,2,97028.0,0,9,3,466.0,...,17.0,2.0,4.0,1.0,1.0,0.0,0.0,1.0,0.0,1
1,77250.0,9.0,19:23,2025-01-17,0,27515.0,0,9,3,215.0,...,9.0,5.0,8.0,1.0,1.0,1.0,0.0,1.0,0.0,1
2,34294.0,3.0,10:20,2025-04-30,2,13810.0,2,2,0,216.0,...,18.0,5.0,8.0,1.0,0.0,0.0,1.0,1.0,0.0,1
3,92041.0,1.0,14:11,2025-02-21,1,10501.0,4,7,2,408.0,...,18.0,5.0,1.0,0.0,1.0,2.0,1.0,1.0,0.0,1
4,71578.0,1.0,04:12,2025-04-11,1,53569.0,2,9,1,209.0,...,18.0,4.0,3.0,0.0,1.0,1.0,0.0,1.0,0.0,1
